In [ ]:
# 1. Install verified packages for T4 GPU
!pip install -q torch transformers datasets accelerate bitsandbytes pandas scikit-learn tqdm

# 2. Verify CUDA & GPU
import torch
print("GPU Available:", torch.cuda.is_available())
print("Device Name  :", torch.cuda.get_device_name(0))

# 3. Create folder structure
!mkdir -p configs data src scripts results

GPU Available: True
Device Name  : Tesla T4


In [ ]:
import os
import re
import json
import torch
import urllib.request
import pandas as pd
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DATA_FILE = "data/pqa_eval_200.json"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# --- 1. Prepare Stratified Dataset ---
def prepare_data():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, "r") as f:
            return json.load(f)

    print("[*] Downloading PubMedQA labeled dataset directly from source...")
    url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json"

    try:
        with urllib.request.urlopen(url) as response:
            raw_data = json.loads(response.read().decode())
    except Exception as e:
        raise RuntimeError(f"Failed to download raw PubMedQA dataset: {e}")

    buckets = {"yes": [], "no": [], "maybe": []}
    for pubid, item in raw_data.items():
        label = item["final_decision"].strip().lower()
        if label in buckets:
            labels = item.get("LABELS", [])
            contexts = item.get("CONTEXTS", [])
            abstract = "\n\n".join([f"{l}: {c}" if l else c for l, c in zip(labels, contexts)])
            buckets[label].append({
                "pubid": int(pubid),
                "question": item["QUESTION"].strip(),
                "abstract": abstract,
                "ground_truth": label
            })

    import random
    rng = random.Random(42)
    for k in buckets:
        rng.shuffle(buckets[k])

    subset = buckets["yes"][:100] + buckets["no"][:66] + buckets["maybe"][:34]
    rng.shuffle(subset)

    with open(DATA_FILE, "w") as f:
        json.dump(subset, f, indent=2)

    print(f"[+] Balanced dataset cached ({len(subset)} samples):", Counter([x["ground_truth"] for x in subset]))
    return subset

dataset = prepare_data()

# --- 2. Load 4-bit NF4 Model on T4 GPU ---
print(f"[*] Loading {MODEL_ID} in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.eval()

# --- 3. Run Inference ---
records = []
print("[*] Running Condition 1 (Clean Baseline) across 200 samples...")

for item in tqdm(dataset, desc="Evaluating"):
    messages = [
        {"role": "system", "content": "You are an expert biomedical information extraction system. Answer strictly using the provided abstract text."},
        {"role": "user", "content": f"Context Abstract:\n{item['abstract']}\n\nQuestion:\n{item['question']}\n\nBased strictly on the abstract provided above, answer the question with exactly one word: \"yes\", \"no\", or \"maybe\". Do not provide additional commentary.\nAnswer:"}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=16, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    raw_output = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    match = re.search(r"\b(yes|no|maybe)\b", raw_output.lower())
    pred = match.group(1) if match else "unparseable"

    records.append({
        "pubid": item["pubid"],
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "raw_output": raw_output,
        "prediction": pred,
        "is_correct": int(pred == item["ground_truth"])
    })

# --- 4. Compute Metrics & Save Outputs ---
df = pd.DataFrame(records)
df.to_csv("results/baseline_predictions.csv", index=False)

acc = accuracy_score(df["ground_truth"], df["prediction"])
macro_f1 = f1_score(df["ground_truth"], df["prediction"], labels=["yes", "no", "maybe"], average="macro")

metrics_df = pd.DataFrame([{
    "condition": "Condition 1 (Clean Baseline)",
    "total_samples": len(df),
    "accuracy": round(float(acc), 4),
    "macro_f1": round(float(macro_f1), 4),
    "unparseable": len(df[~df["prediction"].isin(["yes", "no", "maybe"])])
}])
metrics_df.to_csv("results/baseline_metrics.csv", index=False)

print("\n" + "=" * 50)
print(f"Accuracy : {acc * 100:.2f}%")
print(f"Macro-F1 : {macro_f1 * 100:.2f}%")
print("=" * 50)
print(classification_report(df["ground_truth"], df["prediction"], labels=["yes", "no", "maybe"], digits=4))

[*] Downloading PubMedQA labeled dataset directly from source...
[+] Balanced dataset cached (200 samples): Counter({'yes': 100, 'no': 66, 'maybe': 34})
[*] Loading Qwen/Qwen2.5-7B-Instruct in 4-bit NF4...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (5 outputs left out)

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[*] Running Condition 1 (Clean Baseline) across 200 samples...


Evaluating: 100%|██████████| 200/200 [10:28<00:00,  3.14s/it]


Accuracy : 50.00%
Macro-F1 : 46.20%
              precision    recall  f1-score   support

         yes     0.7500    0.6300    0.6848       100
          no     0.9412    0.2424    0.3855        66
       maybe     0.2121    0.6176    0.3158        34

    accuracy                         0.5000       200
   macro avg     0.6344    0.4967    0.4620       200
weighted avg     0.7216    0.5000    0.5233       200



In [ ]:
from google.colab import files
files.download("results/baseline_predictions.csv")
files.download("results/baseline_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================================================================
# BASELINE EVALUATION PIPELINE (Google Colab - T4 GPU)
# Includes: Direct raw GitHub data fetch + Calibrated Decision Prompt
# ==============================================================================

import os
import re
import json
import torch
import random
import urllib.request
import pandas as pd
from tqdm import tqdm
from collections import Counter
from google.colab import files
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Setup directories
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

DATA_FILE = "data/pqa_eval_200.json"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# --- 1. Prepare Stratified Dataset ---
def prepare_data():
    if os.path.exists(DATA_FILE):
        print(f"[*] Found cached evaluation subset: {DATA_FILE}")
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            return json.load(f)

    print("[*] Downloading PubMedQA labeled dataset directly from GitHub source...")
    url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json"

    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response:
            raw_data = json.loads(response.read().decode())
    except Exception as e:
        raise RuntimeError(f"Failed to download raw PubMedQA dataset: {e}")

    buckets = {"yes": [], "no": [], "maybe": []}
    for pubid, item in raw_data.items():
        label = item.get("final_decision", "").strip().lower()
        if label in buckets:
            labels = item.get("LABELS", [])
            contexts = item.get("CONTEXTS", [])
            abstract = "\n\n".join([f"{l}: {c}" if l else c for l, c in zip(labels, contexts)])
            buckets[label].append({
                "pubid": int(pubid),
                "question": item.get("QUESTION", "").strip(),
                "abstract": abstract,
                "ground_truth": label
            })

    rng = random.Random(42)
    for k in buckets:
        rng.shuffle(buckets[k])

    # 200-sample balanced subset (100 yes, 66 no, 34 maybe)
    subset = buckets["yes"][:100] + buckets["no"][:66] + buckets["maybe"][:34]
    rng.shuffle(subset)

    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(subset, f, indent=2)

    print(f"[+] Balanced dataset cached ({len(subset)} samples):", Counter([x["ground_truth"] for x in subset]))
    return subset

dataset = prepare_data()

# --- 2. Load 4-bit NF4 Model on T4 GPU ---
print(f"[*] Initializing {MODEL_ID} in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.eval()

# --- 3. Run Inference with Calibrated Prompt ---
records = []
print(f"[*] Running Calibrated Condition 1 Baseline across {len(dataset)} samples...")

for item in tqdm(dataset, desc="Evaluating"):
    # Calibrated prompt prevents default 'maybe' collapse on academic hedging
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert biomedical research evaluator. Analyze the provided medical abstract "
                "and identify the authors' primary clinical conclusions."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context Abstract:\n{item['abstract']}\n\n"
                f"Question:\n{item['question']}\n\n"
                "Based on the authors' explicit findings and conclusions in the abstract:\n"
                "- If the authors conclude positively or confirm the hypothesis, answer 'yes'.\n"
                "- If the authors find no effect, refute the premise, or advise against it, answer 'no'.\n"
                "- Only answer 'maybe' if the authors explicitly state the findings are inconclusive or directly conflicting.\n\n"
                "Respond strictly with exactly one word ('yes', 'no', or 'maybe').\n"
                "Answer:"
            )
        }
    ]

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw_output = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    match = re.search(r"\b(yes|no|maybe)\b", raw_output.lower())
    pred = match.group(1) if match else "unparseable"

    records.append({
        "pubid": item["pubid"],
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "raw_output": raw_output,
        "prediction": pred,
        "is_correct": int(pred == item["ground_truth"])
    })

# --- 4. Compute Metrics & Save Outputs ---
preds_path = "results/baseline_predictions.csv"
metrics_path = "results/baseline_metrics.csv"

df = pd.DataFrame(records)
df.to_csv(preds_path, index=False)

acc = accuracy_score(df["ground_truth"], df["prediction"])
macro_f1 = f1_score(df["ground_truth"], df["prediction"], labels=["yes", "no", "maybe"], average="macro")
unparseable_count = len(df[~df["prediction"].isin(["yes", "no", "maybe"])])

metrics_df = pd.DataFrame([{
    "condition": "Condition 1 (Clean Baseline - Calibrated)",
    "total_samples": len(df),
    "accuracy": round(float(acc), 4),
    "macro_f1": round(float(macro_f1), 4),
    "unparseable": unparseable_count
}])
metrics_df.to_csv(metrics_path, index=False)

print("\n" + "=" * 60)
print("            CALIBRATED BASELINE EVALUATION METRICS")
print("=" * 60)
print(f"Accuracy    : {acc * 100:.2f}%")
print(f"Macro-F1    : {macro_f1 * 100:.2f}%")
print(f"Unparseable : {unparseable_count} / {len(df)}")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(df["ground_truth"], df["prediction"], labels=["yes", "no", "maybe"], digits=4))

# --- 5. Download Deliverables ---
print("\n[*] Initiating download of output files...")
files.download(preds_path)
files.download(metrics_path)

[*] Found cached evaluation subset: data/pqa_eval_200.json
[*] Initializing Qwen/Qwen2.5-7B-Instruct in 4-bit NF4...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[*] Running Calibrated Condition 1 Baseline across 200 samples...


Evaluating: 100%|██████████| 200/200 [11:56<00:00,  3.58s/it]


            CALIBRATED BASELINE EVALUATION METRICS
Accuracy    : 62.50%
Macro-F1    : 58.67%
Unparseable : 0 / 200

Classification Report:
              precision    recall  f1-score   support

         yes     0.7849    0.7300    0.7565       100
          no     0.8684    0.5000    0.6346        66
       maybe     0.2754    0.5588    0.3689        34

    accuracy                         0.6250       200
   macro avg     0.6429    0.5963    0.5867       200
weighted avg     0.7259    0.6250    0.6504       200


[*] Initiating download of output files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================================================================
# PHASE IV: 4-CONDITION PERTURBATION & HALLUCINATION BENCHMARK (Single-Cell)
# Evaluates:
#   - Condition 2: No-Context (Closed-Book / Parametric Prior)
#   - Condition 3: Inverted / Counterfactual Context (Faithfulness Test)
#   - Condition 4: Irrelevant Distractor Context (Groundedness / Noise Test)
# Calculates: CFR (Context Faithfulness Rate), IHR (Intrinsic Hallucination Rate), PAR
# ==============================================================================

# Ensure bitsandbytes is updated and installed in Colab's current session first
import sys
!{sys.executable} -m pip install -q -U bitsandbytes accelerate

import json
import os
import random
import re
import urllib.request
from google.colab import files
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DATA_FILE = "data/pqa_eval_200.json"
RESULTS_DIR = "results"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

os.makedirs("data", exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 1. Ensure dataset exists, download and prepare if missing
if not os.path.exists(DATA_FILE):
  print("[*] Evaluation dataset not found. Downloading and preparing...")
  url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json"
  try:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
      raw_data = json.loads(response.read().decode())
  except Exception as e:
    raise RuntimeError(f"Failed to download raw PubMedQA dataset: {e}")

  buckets = {"yes": [], "no": [], "maybe": []}
  for pubid, item in raw_data.items():
    label = item.get("final_decision", "").strip().lower()
    if label in buckets:
      labels = item.get("LABELS", [])
      contexts = item.get("CONTEXTS", [])
      abstract = "\n\n".join([f"{l}: {c}" if l else c for l, c in zip(labels, contexts)])
      buckets[label].append({
          "pubid": int(pubid),
          "question": item.get("QUESTION", "").strip(),
          "abstract": abstract,
          "ground_truth": label
      })

  rng = random.Random(42)
  for k in buckets:
    rng.shuffle(buckets[k])

  # 200-sample balanced subset (100 yes, 66 no, 34 maybe)
  dataset = buckets["yes"][:100] + buckets["no"][:66] + buckets["maybe"][:34]
  rng.shuffle(dataset)

  with open(DATA_FILE, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)
  print(f"[+] Balanced dataset prepared and saved to {DATA_FILE}")
else:
  with open(DATA_FILE, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# 2. Ensure model and tokenizer are initialized
if 'model' not in globals() or 'tokenizer' not in globals():
  print(f"[*] Model/Tokenizer not detected in environment. Initializing {MODEL_ID} in 4-bit NF4...")
  bnb_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_use_double_quant=True,
      bnb_4bit_compute_dtype=torch.bfloat16
  )
  tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
  model = AutoModelForCausalLM.from_pretrained(
      MODEL_ID,
      quantization_config=bnb_config,
      device_map="auto",
      torch_dtype=torch.bfloat16
  )
  model.eval()
else:
  print("[*] Model and Tokenizer found in active environment.")

# Load baseline predictions if available for delta comparison
baseline_csv_path = os.path.join(RESULTS_DIR, "baseline_predictions.csv")
baseline_preds = {}
if os.path.exists(baseline_csv_path):
  b_df = pd.read_csv(baseline_csv_path)
  baseline_preds = dict(zip(b_df["pubid"], b_df["prediction"]))


# 3. Perturbation Engine Class
class PerturbationEngine:

  def __init__(self, seed: int = 42):
    self.seed = seed
    self.rng = random.Random(seed)
    self.inversion_patterns = [
        (
            r"\bis significantly associated with\b",
            "is not significantly associated with",
        ),
        (
            r"\bis not significantly associated with\b",
            "is significantly associated with",
        ),
        (r"\bwas significantly associated with\b", "was not associated with"),
        (r"\bwas associated with\b", "was not associated with"),
        (r"\bis associated with\b", "is not associated with"),
        (r"\bshowed significant improvement\b", "showed no improvement"),
        (r"\bdemonstrated efficacy\b", "failed to demonstrate efficacy"),
        (r"\bsignificantly reduced\b", "did not reduce"),
        (r"\bsignificantly increased\b", "did not increase"),
        (r"\bcan be recommended\b", "cannot be recommended"),
        (r"\bis effective\b", "is ineffective"),
        (r"\bis safe\b", "is unsafe"),
        (r"\bplays an important role\b", "does not play a role"),
        (r"\bcorrelates with\b", "does not correlate with"),
        (r"\bhas a beneficial effect\b", "has no beneficial effect"),
    ]

  def generate_condition_2(self, item):
    return {
        "pubid": item["pubid"],
        "question": item["question"],
        "abstract": "",
        "ground_truth": item["ground_truth"],
        "target_comparison": item["ground_truth"],
        "condition": "Condition 2 (No-Context)",
    }

  def generate_condition_3(self, item):
    abstract = item["abstract"]
    inverted_abstract = abstract
    for pattern, replacement in self.inversion_patterns:
      inverted_abstract = re.sub(
          pattern, replacement, inverted_abstract, flags=re.IGNORECASE
      )

    if inverted_abstract == abstract:
      inverted_abstract = (
          "CONCLUSIONS: Contrary to prior findings, results demonstrate no"
          f" significant clinical effect or validity.\n\n{abstract}"
      )

    gt = item["ground_truth"]
    inverted_target = (
        "no" if gt == "yes" else ("yes" if gt == "no" else "maybe")
    )
    return {
        "pubid": item["pubid"],
        "question": item["question"],
        "abstract": inverted_abstract,
        "ground_truth": gt,
        "target_comparison": inverted_target,
        "condition": "Condition 3 (Inverted Context)",
    }

  def generate_condition_4(self, dataset):
    n = len(dataset)
    shift = self.rng.randint(1, n - 1)
    items = []
    for i, item in enumerate(dataset):
      unrelated_item = dataset[(i + shift) % n]
      items.append({
          "pubid": item["pubid"],
          "question": item["question"],
          "abstract": unrelated_item["abstract"],
          "ground_truth": item["ground_truth"],
          "target_comparison": "maybe",  # Expected behavior: evidence does not support answering
          "condition": "Condition 4 (Distractor Context)",
      })
    return items


engine = PerturbationEngine(seed=42)

# 4. Helper function for running batch inference
def run_condition_eval(eval_items, condition_name):
  records = []
  print(f"\n[*] Running {condition_name} ({len(eval_items)} samples)...")

  for item in tqdm(eval_items, desc=f"{condition_name}"):
    context_block = (
        f"Context Abstract:\n{item['abstract']}\n\n"
        if item["abstract"]
        else ""
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert biomedical research evaluator. Analyze the"
                " provided clinical question and context to identify the"
                " explicit medical conclusions."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{context_block}"
                f"Question:\n{item['question']}\n\n"
                "Based on the authors' explicit findings:\n"
                "- If the evidence confirms the hypothesis/premise, answer"
                " 'yes'.\n"
                "- If the evidence refutes the premise or shows no effect,\n"
                " answer 'no'.\n"
                "- If the evidence is inconclusive, missing, or unrelated,\n"
                " answer 'maybe'.\n\n"
                "Respond strictly with exactly one word ('yes', 'no', or"
                " 'maybe').\n"
                "Answer:"
            ),
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
      output = model.generate(
          **inputs,
          max_new_tokens=16,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
      )

    raw_output = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()
    match = re.search(r"\b(yes|no|maybe)\b", raw_output.lower())
    pred = match.group(1) if match else "unparseable"

    records.append({
        "pubid": item["pubid"],
        "condition": condition_name,
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "target_comparison": item["target_comparison"],
        "raw_output": raw_output,
        "prediction": pred,
        "matches_target": int(pred == item["target_comparison"]),
        "baseline_prediction": baseline_preds.get(item["pubid"], "N/A"),
    })
  return records


# 5. Generate Perturbed Datasets
c2_dataset = [engine.generate_condition_2(x) for x in dataset]
c3_dataset = [engine.generate_condition_3(x) for x in dataset]
c4_dataset = engine.generate_condition_4(dataset)

# 6. Execute Evaluations
results_c2 = run_condition_eval(c2_dataset, "Condition 2 (No-Context)")
results_c3 = run_condition_eval(c3_dataset, "Condition 3 (Inverted Context)")
results_c4 = run_condition_eval(c4_dataset, "Condition 4 (Distractor Context)")

all_perturbation_results = results_c2 + results_c3 + results_c4
df_all = pd.DataFrame(all_perturbation_results)
perturbations_csv_path = os.path.join(
    RESULTS_DIR, "perturbation_predictions.csv"
)
df_all.to_csv(perturbations_csv_path, index=False)

# 7. Compute Research Metrics (CFR, IHR, PAR)
df_c2 = pd.DataFrame(results_c2)
df_c3 = pd.DataFrame(results_c3)
df_c4 = pd.DataFrame(results_c4)

# PAR: Parametric Alignment Rate (Condition 2 matches true label)
par_score = accuracy_score(df_c2["ground_truth"], df_c2["prediction"])

# CFR: Context Faithfulness Rate (Condition 3 matches inverted counterfactual label)
cfr_score = accuracy_score(df_c3["target_comparison"], df_c3["prediction"])

# IHR: Intrinsic Hallucination Rate (Condition 4 asserts yes/no despite irrelevant context)
ihr_count = len(df_c4[df_c4["prediction"].isin(["yes", "no"])])
ihr_score = ihr_count / len(df_c4)

# Macro F1 across conditions
f1_c2 = f1_score(
    df_c2["ground_truth"],
    df_c2["prediction"],
    labels=["yes", "no", "maybe"],
    average="macro",
)
f1_c3 = f1_score(
    df_c3["target_comparison"],
    df_c3["prediction"],
    labels=["yes", "no", "maybe"],
    average="macro",
)
f1_c4 = f1_score(
    df_c4["target_comparison"],
    df_c4["prediction"],
    labels=["yes", "no", "maybe"],
    average="macro",
)

summary_metrics = pd.DataFrame([
    {
        "Condition": "Condition 1 (Clean Baseline)",
        "Target Metric": "Accuracy vs Ground Truth",
        "Score": 0.6250,
        "Macro_F1": 0.5867,
    },
    {
        "Condition": "Condition 2 (No-Context)",
        "Target Metric": "PAR (Parametric Alignment)",
        "Score": round(float(par_score), 4),
        "Macro_F1": round(float(f1_c2), 4),
    },
    {
        "Condition": "Condition 3 (Inverted Context)",
        "Target Metric": "CFR (Context Faithfulness)",
        "Score": round(float(cfr_score), 4),
        "Macro_F1": round(float(f1_c3), 4),
    },
    {
        "Condition": "Condition 4 (Distractor Context)",
        "Target Metric": "IHR (Intrinsic Hallucination)",
        "Score": round(float(ihr_score), 4),
        "Macro_F1": round(float(f1_c4), 4),
    },
])

summary_csv_path = os.path.join(RESULTS_DIR, "perturbation_metrics_summary.csv")
summary_metrics.to_csv(summary_csv_path, index=False)

print("\n" + "=" * 65)
print("       PHASE IV: PERTURBATION EXPERIMENTAL MATRIX RESULTS")
print("=" * 65)
print(summary_metrics.to_string(index=False))
print("=" * 65)
print(f"[*] Parametric Alignment Rate (PAR)    : {par_score * 100:.2f}%")
print(f"[*] Context Faithfulness Rate (CFR)    : {cfr_score * 100:.2f}%")
print(f"[*] Intrinsic Hallucination Rate (IHR) : {ihr_score * 100:.2f}%")
print("=" * 65)

# 8. Safe download of files
if os.path.exists(perturbations_csv_path):
  files.download(perturbations_csv_path)
if os.path.exists(summary_csv_path):
  files.download(summary_csv_path)

[*] Evaluation dataset not found. Downloading and preparing...
[+] Balanced dataset prepared and saved to data/pqa_eval_200.json
[*] Model/Tokenizer not detected in environment. Initializing Qwen/Qwen2.5-7B-Instruct in 4-bit NF4...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (14 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# Create destination folder
os.makedirs("papers/figures", exist_ok=True)

conditions = [
    "Condition 1\n(Clean Baseline)",
    "Condition 2\n(No-Context / PAR)",
    "Condition 3\n(Inverted / CFR)",
    "Condition 4\n(Distractor / IHR)",
]
scores = [62.50, 17.50, 31.50, 0.00]
macro_f1s = [58.67, 10.39, 24.82, 33.33]

x = np.arange(len(conditions))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4.8), dpi=300)
rects1 = ax.bar(
    x - width / 2,
    scores,
    width,
    label="Target Metric Score (%)",
    color="#2b5c8f",
    edgecolor="black",
    alpha=0.9,
)
rects2 = ax.bar(
    x + width / 2,
    macro_f1s,
    width,
    label="Macro-F1 (x100)",
    color="#d95f02",
    edgecolor="black",
    alpha=0.9,
)

ax.set_ylabel("Percentage / Score (%)", fontsize=11, fontweight="bold")
ax.set_title(
    "Evaluation Across 4-Condition Perturbation Matrix (PubMedQA N=200)",
    fontsize=12,
    fontweight="bold",
    pad=12,
)
ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=9.5)
ax.set_ylim(0, 75)
ax.legend(frameon=True, loc="upper right")


def autolabel(rects):
  for rect in rects:
    h = rect.get_height()
    ax.annotate(
        f"{h:.1f}%",
        xy=(rect.get_x() + rect.get_width() / 2, h),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8.5,
        fontweight="bold",
    )


autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.savefig("papers/figures/perturbation_matrix_evaluation.png", dpi=300)
plt.savefig("papers/figures/perturbation_matrix_evaluation.pdf")
plt.close()
print(
    "[+] Successfully created papers/figures/ with .png and .pdf visual"
    " artifacts."
)

[+] Successfully created papers/figures/ with .png and .pdf visual artifacts.
